In [1]:
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
#from imblearn.over_sampling import SMOTE
#from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB
import joblib


Note: This version does not perform CV for the positive data. Applies 5-fold undersampling for neg data and saves each model

In [2]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [3]:
root_folder='../data/train_data/embedding/birdnet/'
df_pos = get_embedding_birdnet(root_folder, 1)
df_pos["filename"] = df_pos["embed_name"].str.split("-").str[0].str[:-1]


In [4]:
df_pos.filename = df_pos.filename

In [5]:
root_folder='../data/test_data_4/embedding/'
df_neg = get_embedding_birdnet(root_folder, 0)
df_neg["filename"] = df_neg["embed_name"].str.split("_").str[0]


In [6]:
df_pos['target'] = 1
df_neg['target'] = 0

In [7]:
df_neg["recording_id"] = df_neg["embed_name"].str.split("_").str[0]

In [8]:
df_neg["recording_id"] = df_neg["recording_id"].astype(int)

In [9]:
df_neg_full = pd.read_csv('./script_api/xc_non_Eurasian_downsampled.csv')

In [10]:
df_neg = df_neg.merge(
    df_neg_full,
    on="recording_id",
    how="left"
)

In [11]:
df_neg.user.nunique()

446

In [12]:
df_neg.sample(3)

,embed_name,embedding,filename,target,recording_id,mic,species,country,date,duration,user,recording_url
2307,929610_3.birdnet.embeddings.txt,"[0.0, 0.4948577, 0.0, 0.9752094, 0.4615077, 0....",929610,0,929610,NaN,Common Sandpiper,France,2024-08-05,0:29,Martin Billard,https://xeno-canto.org/929610
751,1055194_2.birdnet.embeddings.txt,"[0.2527936, 1.5136942, 0.6921187, 0.0, 0.01826...",1055194,0,1055194,Primo EM 272,Little Stint,France,2025-09-07,0:16,Stanislas Wroza,https://xeno-canto.org/1055194
1878,900593_4.birdnet.embeddings.txt,"[0.07281774, 2.3682098, 0.21401614, 0.46149197...",900593,0,900593,Hi-Sound Stereo,Spotted Redshank,Sweden,2024-05-05,0:10,Jochem verweij,https://xeno-canto.org/900593


In [13]:
n_folds = 5
rng = np.random.default_rng(42)

# Downsample each user independently so their records divide evenly across folds
df_neg = df_neg.copy()
df_neg["user"] = df_neg["user"].fillna("<missing_user>")

fold_parts = []

for user, user_df in df_neg.groupby("user", sort=False):
    user_df = user_df.sample(frac=1, random_state=42).reset_index(drop=True)

    usable_count = (len(user_df) // n_folds) * n_folds
    user_df = user_df.iloc[:usable_count].copy()

    # Assign records round-robin to distribute each user's rows uniformly
    user_df["fold"] = np.arange(len(user_df)) % n_folds
    fold_parts.append(user_df)

df_neg = pd.concat(fold_parts, ignore_index=True)

# Shuffle the final dataframe
df_neg = df_neg.sample(frac=1, random_state=42).reset_index(drop=True)

# Check user distribution across folds
user_fold_counts = pd.crosstab(df_neg["user"], df_neg["fold"])
display(user_fold_counts)

fold,0,1,2,3,4
user,,,,,
Aitor Zabala,4,4,4,4,4
Aku Kalliomäki,9,9,9,9,9
Aladdin,1,1,1,1,1
Alan Dalton,2,2,2,2,2
Albert Lastukhin,7,7,7,7,7
...,...,...,...,...,...
david thorns,1,1,1,1,1
etienne.roland@gmail.com,1,1,1,1,1
florent yvert,2,2,2,2,2


In [14]:
# Preserve the restored negative dataframe before adding fold assignments
df_neg_original = df_neg.copy()

# Assign every record to a fold; no records are discarded
fold_parts = []

for user, user_df in df_neg_original.groupby("user", sort=False):
    user_df = user_df.sample(frac=1, random_state=42).copy()
    user_df["fold"] = np.arange(len(user_df)) % n_folds
    fold_parts.append(user_df)

df_neg = (
    pd.concat(fold_parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(f"Users retained: {df_neg['user'].nunique()}")
print(f"Records retained: {len(df_neg)}")
display(pd.crosstab(df_neg["user"], df_neg["fold"]))

Users retained: 125
Records retained: 2175


fold,0,1,2,3,4
user,,,,,
Aitor Zabala,4,4,4,4,4
Aku Kalliomäki,9,9,9,9,9
Aladdin,1,1,1,1,1
Alan Dalton,2,2,2,2,2
Albert Lastukhin,7,7,7,7,7
...,...,...,...,...,...
david thorns,1,1,1,1,1
etienne.roland@gmail.com,1,1,1,1,1
florent yvert,2,2,2,2,2


In [15]:
df_neg["user"].nunique()

125

In [16]:
df_neg.filename = df_neg.user

In [30]:
fold=2
df_neg_fold = df_neg[df_neg.fold==fold]
df_neg_fold = df_neg_fold.drop("fold", axis=1)

In [31]:
from sklearn.model_selection import GroupKFold, cross_val_score

# Combine and shuffle
df_combined = pd.concat([df_pos, df_neg_fold], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=22).reset_index(drop=True)

# Features, labels, groups
X = np.vstack(df_combined["embedding"].values)
y = df_combined["target"].values
groups = df_combined["filename"].values

# 5-fold grouped CV
cv = GroupKFold(n_splits=5)

Perform 5-fold split for the negative data only

In [32]:
from sklearn.svm import SVC

model = SVC(kernel="rbf", cache_size=500)

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")

ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)


print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9487 ± 0.0221
AP: 0.9925 ± 0.0080


SVM

In [21]:
import joblib
from sklearn.svm import SVC

# Run for all 5 folds
for fold in range(5):

    print(f"\nTraining model for fold {fold}...")

    # Select negatives for this fold
    df_neg_fold = df_neg[df_neg["fold"] == fold].copy()
    df_neg_fold = df_neg_fold.drop("fold", axis=1)

    # Combine positives and negatives
    df_combined = pd.concat(
        [df_pos, df_neg_fold],
        ignore_index=True
    )

    # Shuffle
    df_combined = df_combined.sample(
        frac=1,
        random_state=22
    ).reset_index(drop=True)

    # Features and labels
    X = np.vstack(df_combined["embedding"].values)
    y = df_combined["target"].values

    # Create SVC
    model = SVC(
        kernel="rbf",
        probability=True,
        random_state=22
    )

    # Train on full dataset for this fold
    model.fit(X, y)

    # Save model
    model_path = f"svc_model_negfold_{fold}.joblib"
    joblib.dump(model, model_path)

    print(f"Model saved as: {model_path}")

print("\nAll 5 models trained and saved.")




Training model for fold 0...
Model saved as: svc_model_negfold_0.joblib

Training model for fold 1...
Model saved as: svc_model_negfold_1.joblib

Training model for fold 2...
Model saved as: svc_model_negfold_2.joblib

Training model for fold 3...
Model saved as: svc_model_negfold_3.joblib

Training model for fold 4...
Model saved as: svc_model_negfold_4.joblib

All 5 models trained and saved.


In [22]:
train_accuracy = model.score(X, y)
print(f"Train accuracy: {train_accuracy:.4f}")

Train accuracy: 1.0000


Random Forest

In [23]:
# Run for all 5 negative folds
for fold in range(5):

    print(f"\nTraining model for fold {fold}...")

    # Select negatives for this fold
    df_neg_fold = df_neg[df_neg["fold"] == fold].copy()
    df_neg_fold = df_neg_fold.drop("fold", axis=1)

    # Combine positives and negatives
    df_combined = pd.concat(
        [df_pos, df_neg_fold],
        ignore_index=True
    )

    # Shuffle
    df_combined = df_combined.sample(
        frac=1,
        random_state=22
    ).reset_index(drop=True)

    # Features and labels
    X = np.vstack(df_combined["embedding"].values)
    y = df_combined["target"].values

    # Create RFC
    model = RFC(n_jobs = -1)

    # Train on full dataset for this fold
    model.fit(X, y)

    # Save model
    model_path = f"rfc_model_negfold_{fold}.joblib"
    joblib.dump(model, model_path)

    print(f"Model saved as: {model_path}")

print("\nAll 5 models trained and saved.")


Training model for fold 0...
Model saved as: rfc_model_negfold_0.joblib

Training model for fold 1...
Model saved as: rfc_model_negfold_1.joblib

Training model for fold 2...
Model saved as: rfc_model_negfold_2.joblib

Training model for fold 3...
Model saved as: rfc_model_negfold_3.joblib

Training model for fold 4...
Model saved as: rfc_model_negfold_4.joblib

All 5 models trained and saved.


XGBoost

In [24]:


# Run for all 5 negative folds
for fold in range(5):

    print(f"\nTraining model for fold {fold}...")

    # Select negatives for this fold
    df_neg_fold = df_neg[df_neg["fold"] == fold].copy()
    df_neg_fold = df_neg_fold.drop("fold", axis=1)

    # Combine positives and negatives
    df_combined = pd.concat(
        [df_pos, df_neg_fold],
        ignore_index=True
    )

    # Shuffle
    df_combined = df_combined.sample(
        frac=1,
        random_state=22
    ).reset_index(drop=True)

    # Features and labels
    X = np.vstack(df_combined["embedding"].values)
    y = df_combined["target"].values

    # Create RFC
    model = XGB.XGBClassifier(objective='binary:logistic')

    # Train on full dataset for this fold
    model.fit(X, y)

    # Save model
    model_path = f"xgb_model_negfold_{fold}.joblib"
    joblib.dump(model, model_path)

    print(f"Model saved as: {model_path}")



Training model for fold 0...
Model saved as: xgb_model_negfold_0.joblib

Training model for fold 1...
Model saved as: xgb_model_negfold_1.joblib

Training model for fold 2...
Model saved as: xgb_model_negfold_2.joblib

Training model for fold 3...
Model saved as: xgb_model_negfold_3.joblib

Training model for fold 4...
Model saved as: xgb_model_negfold_4.joblib


This part is from the old script, where 5-fold CV is done on the positives as well

In [25]:
import os
import joblib
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.metrics import confusion_matrix

cv = GroupKFold(n_splits=5)

dataset = "ds1"

accuracies = []
aps = []
confusion_matrices = []

os.makedirs("saved_models", exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = SVC(kernel="rbf", cache_size=500)

    model.fit(X_train, y_train)

    # Save model
    joblib.dump(model, f"saved_models/{dataset}_svm_fold_{fold}.joblib")

    # Evaluate
    y_pred = model.predict(X_test)
    y_score = model.decision_function(X_test)

    accuracies.append(accuracy_score(y_test, y_pred))
    aps.append(average_precision_score(y_test, y_score))
    cm = confusion_matrix(y_test, y_pred)
    confusion_matrices.append(cm)

    np.savetxt(
        f"saved_models/confusion_matrix_fold_{fold}.csv",
        cm,
        fmt="%d",
        delimiter=","
    )

print("Accuracy per fold:", np.round(accuracies, 4))
print(f"Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies, ddof=1):.4f}")

print("AP per fold:", np.round(aps, 4))
print(f"AP: {np.mean(aps):.4f} ± {np.std(aps, ddof=1):.4f}")


Accuracy per fold: [0.9628 0.9519 0.9519 0.9893 0.9572]
Accuracy: 0.9626 ± 0.0156
AP per fold: [0.9989 0.9847 0.999  1.     0.9985]
AP: 0.9962 ± 0.0064


In [26]:
#plt.plot(recall, precision, marker='.')
#plt.xlabel('Recall')
#plt.ylabel('Precision')
#plt.title('Precision-Recall Curve')
#plt.show()